## **Generación Etiquetas: Answerability**

In [ ]:
# !pip install -q cohere

In [ ]:
def answerability(new_question: str, context: str, conversation_history: str = '') -> str:
    """Función que genera una etiqueta (ANSWERABLE, UNANSWERABLE, PARTIAL) para describir a la nueva pregunta con respecto a la conversación previa en caso de que haya."""

    base_prompt = """You are a precision-focused evaluation model. Your task is to analyze three inputs:
    1. **Context**: Provided information that may contain relevant facts or data.
    2. **Conversation History**: Previous questions and answers in the dialogue. Sometimes there is no previs dialogue.
    3. **New Question**: The latest query from the user.

    Your job is to determine **if the new question can be answered** using *only* the given context and conversation history.
    You should first defend each posture and make the case on why the question can totally, can partially and cannot be answered.
    After analyzing each posture, provide your final answer. Remember that the only information you should take into condideration is the provided one.

    **Output Format:**
    Provide your response in the following two-part format. Do not include any other text before, after, or between these parts.

    **Label:** [ANSWERABLE/UNANSWERABLE/PARTIAL/CONVERSATIONAL]
    **Explanation:** [A single, concise sentence explaining the reason for the label, based solely on the provided information.]
    **Other Postures:** ["Label", "Explanation"]

    **Label Definitions:**
    - **"ANSWERABLE"**: A question is ANSWERABLE when the provided context contains sufficient, explicit, and specific information to formulate a direct and complete response that addresses all parts of the query without requiring external knowledge or inference beyond simple clarification of pronouns or coreferences resolvable within the text.
    - **"UNANSWERABLE"**: A question is UNANSWERABLE when the provided context lacks the specific, objective information required to formulate any correct response, either because the question asks about details not mentioned, is too vague or ambiguous to interpret within the context, or seeks subjective judgment or external knowledge.
    - **"PARTIAL"**: A question is PARTIALLY answerable when the provided context contains some relevant information that addresses the general topic or a subset of the query, but lacks the specific details, evidence, or completeness required to provide a full and definitive answer.
    - **"CONVERSATIONAL"**: A CONVERSATIONAL label is applied when the user's input is not a substantive question seeking factual information from the context, but rather a social remark, procedural utterance, or conversational turn meant to manage the interaction itself.

    **Considerations:**
    - Unlike CONVERSATIONAL, an UNANSWERABLE input is a genuine (if flawed) question seeking factual information. Unlike PARTIAL, the context provides *no* relevant information that directly addresses the core query.
    - Unlike ANSWERABLE, a PARTIAL context cannot yield a complete answer. Unlike UNANSWERABLE, a PARTIAL context does provide *some* directly relevant factual information that moves toward an answer.
    - Unlike UNANSWERABLE or PARTIAL, a CONVERSATIONAL input is not attempting to query the contextual information at all. Its purpose is fundamentally different.


    **Critical Rules:**
    - **Do not answer the question itself.** Your explanation must only justify the label, not provide the answer.
    - **Do not generate any part of an answer.**
    - **Base your decision and explanation strictly on the provided inputs.**
    - **Keep the explanation brief and factual.**

**Inputs:**
"""

    if(conversation_history != ''):
        prompt = base_prompt + f"## Context: \n{context}\n## Conversation History: \n{conversation_history}\n## New Question: \n{new_question}\n\n**Output:** [Yes/Partial/No]"
    else:
        prompt = base_prompt + f"## Context: \n{context}\n## New Question: \n{new_question}\n\n**Output:** [Yes/Partial/No]"

    # Aquí vaunafuncion de inferencia, yo use una personalizadapara DeepSeek
    # el resultado se debeasignar a result
    # result = inferencia(prompt)
    result = "" # borrar esta línea una vez hecho el cambio

    if 'UNANSWERABLE' in text.split("\n")[0]:
        return 'UNANSWERABLE'
    elif 'ANSWERABLE' in text.split("\n")[0]:
        return 'ANSWERABLE'
    elif 'PARTIAL' in text.split("\n")[0]:
        return 'PARTIAL'
    else:
        return 'CONVERSATIONAL'

In [ ]:
import json
import cohere
from typing import Dict
import time
import random

# ==========================
#   COHERE SETUP
# ==========================
COHERE_API_KEY = "Wm1o9FWweEEVnfy7N2GkOwc7dIKIvK5JDe2SLpKy"
COHERE_MODEL = "command-a-03-2025"

co = cohere.Client(COHERE_API_KEY)

# ==========================
#   UTILIDADES TASK A / C
# ==========================
def get_current_question(task: Dict) -> str:
    # TaskAC usa directamente "question"
    return task.get("question", "")

def format_conversation_history(task: Dict) -> str:
    # Task A / C NO tiene historial
    return ""

def format_context(task: Dict) -> str:
    ctxs = task.get("contexts", [])
    return "\n\n".join(c.get("text", "") for c in ctxs)

# ==========================
#   ANSWERABILITY (COHERE)
# ==========================
def answerability(new_question: str, context: str, conversation_history: str = '') -> str:
    """Genera ANSWERABLE / PARTIAL / UNANSWERABLE usando Cohere Chat"""

    base_prompt = """You are a precision-focused evaluation model. Your task is to analyze three inputs:
1. **Context**: Provided information that may contain relevant facts or data.
2. **Conversation History**: Previous questions and answers in the dialogue.
3. **New Question**: The latest query from the user.

Your job is to determine **if the new question can be answered** using *only* the given context and conversation history. You have no access to external information.

**Output Format:**
Provide your response in the following two-part format.

**Label:** [Yes/Partial/No]
**Explanation:** [One concise sentence justifying the label.]

**Critical Rules:**
- Do not answer the question.
- Use only the provided context.
"""

    prompt = (
        base_prompt
        + f"## Context:\n{context}\n"
          f"## New Question:\n{new_question}\n\n"
          "**Output:**"
    )

    for attempt in range(5):
        try:
            response = co.chat(
                model=COHERE_MODEL,
                message=prompt,
                temperature=0.0
            )
            break
        except Exception as e:
            if attempt == 4:
                raise e
            time.sleep(2 ** attempt + random.uniform(0, 1))

    result = response.text.strip()
    first_line = result.split("\n")[0]

    if 'Yes' in first_line:
        return 'ANSWERABLE'
    elif 'Partial' in first_line:
        return 'PARTIAL'
    elif 'No' in first_line:
        return 'UNANSWERABLE'
    else:
        return 'UNANSWERABLE'

# ==========================
#   PROCESAR rag_taskAC.jsonl
# ==========================
input_path = "rag_taskAC.jsonl"
output_path = "rag_taskAC_with_answerability.jsonl"

with open(input_path, "r", encoding="utf-8") as fin, \
     open(output_path, "w", encoding="utf-8") as fout:

    for line in fin:
        task = json.loads(line)

        new_question = get_current_question(task)
        context = format_context(task)

        label = answerability(
            new_question=new_question,
            context=context,
            conversation_history=""
        )

        task["answerability"] = label
        fout.write(json.dumps(task, ensure_ascii=False) + "\n")

print("✅ Procesamiento terminado")
print(f"📄 Archivo generado: {output_path}")


✅ Procesamiento terminado
📄 Archivo generado: rag_taskAC_with_answerability.jsonl


## **Join de los documentos del Retrival**

In [ ]:
import json

input_path = "IIMAS-RAG_taskA.jsonl"
output_path = "IIMAS-RAG_taskA_top3_contexts.jsonl"

with open(input_path, "r", encoding="utf-8") as fin, \
     open(output_path, "w", encoding="utf-8") as fout:

    for line in fin:
        task = json.loads(line)

        # Si existe el campo contexts y es una lista
        if "contexts" in task and isinstance(task["contexts"], list):
            # Nos quedamos solo con los primeros 3 documentos (izq → der)
            task["contexts"] = task["contexts"][:3]

        fout.write(json.dumps(task, ensure_ascii=False) + "\n")

print("✅ Listo")
print(f"📄 Archivo generado: {output_path}")


✅ Listo
📄 Archivo generado: IIMAS-RAG_taskA_top3_contexts.jsonl


In [ ]:
import json

# ==========================
#   PATHS
# ==========================
MAIN_INPUT = "IIMAS-RAG_taskA_top3_contexts.jsonl"
MAIN_OUTPUT = "IIMAS-RAG_taskA_top3_contexts_with_text.jsonl"

SOURCE_FILES = {
    "clapnq": "clapnq.jsonl",
    "ibmcloud": "cloud.jsonl",
    "fiqa": "fiqa.jsonl",
    "govt": "govt.jsonl",
}

# ==========================
#   1. INDEXAR CADA FUENTE
#   (id -> text)
# ==========================
source_indexes = {}

for collection, path in SOURCE_FILES.items():
    print(f"📚 Indexando {collection}...")
    idx = {}
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            doc = json.loads(line)

            # TODAS las fuentes traen estas variables
            doc_id = doc.get("id") or doc.get("document_id") or doc.get("_id")
            text = doc.get("text", "")

            if doc_id is not None:
                idx[str(doc_id)] = text

    source_indexes[collection] = idx
    print(f"   ↳ {len(idx)} documentos indexados")

# ==========================
#   2. AGREGAR TEXT AL MAIN
# ==========================
found, missing = 0, 0

with open(MAIN_INPUT, "r", encoding="utf-8") as fin, \
     open(MAIN_OUTPUT, "w", encoding="utf-8") as fout:

    for line in fin:
        task = json.loads(line)
        collection = task.get("Collection")

        if collection not in source_indexes:
            raise ValueError(f"Colección desconocida: {collection}")

        index = source_indexes[collection]

        if "contexts" in task and isinstance(task["contexts"], list):
            for ctx in task["contexts"]:
                doc_id = ctx.get("document_id")
                if doc_id is not None and str(doc_id) in index:
                    ctx["text"] = index[str(doc_id)]
                    found += 1
                else:
                    ctx["text"] = ""
                    missing += 1

        fout.write(json.dumps(task, ensure_ascii=False) + "\n")

print("\n🎉 Proceso terminado")
print(f"📄 Archivo generado: {MAIN_OUTPUT}")
print(f"✅ Contextos con texto: {found}")
print(f"❌ Contextos sin texto: {missing}")


📚 Indexando clapnq...
   ↳ 183408 documentos indexados
📚 Indexando ibmcloud...
   ↳ 72442 documentos indexados
📚 Indexando fiqa...
   ↳ 61022 documentos indexados
📚 Indexando govt...
   ↳ 49607 documentos indexados

🎉 Proceso terminado
📄 Archivo generado: IIMAS-RAG_taskA_top3_contexts_with_text.jsonl
✅ Contextos con texto: 1521
❌ Contextos sin texto: 0


In [ ]:
import json

# ==========================
#   PATHS
# ==========================
contexts_path = "IIMAS-RAG_taskA_top3_contexts_with_text.jsonl"
tasks_path = "rag_taskAC_with_answerability.jsonl"
output_path = "rag_taskAC_with_answerability_and_contexts.jsonl"

# ==========================
#   1. INDEXAR CONTEXTS POR task_id
# ==========================
taskid_to_contexts = {}

with open(contexts_path, "r", encoding="utf-8") as f:
    for line in f:
        obj = json.loads(line)
        task_id = obj.get("task_id")
        contexts = obj.get("contexts")

        if task_id is not None and contexts is not None:
            taskid_to_contexts[str(task_id)] = contexts

print(f"📚 Contexts indexados: {len(taskid_to_contexts)}")

# ==========================
#   2. MERGE EN rag_taskAC
# ==========================
merged, missing = 0, 0

with open(tasks_path, "r", encoding="utf-8") as fin, \
     open(output_path, "w", encoding="utf-8") as fout:

    for line in fin:
        task = json.loads(line)
        task_id = task.get("task_id")

        if task_id is not None and str(task_id) in taskid_to_contexts:
            task["contexts"] = taskid_to_contexts[str(task_id)]
            merged += 1
        else:
            missing += 1

        fout.write(json.dumps(task, ensure_ascii=False) + "\n")

print("🎉 Merge terminado")
print(f"✅ Tasks con contexts agregados: {merged}")
print(f"❌ Tasks sin match de contexts: {missing}")
print(f"📄 Archivo generado: {output_path}")


📚 Contexts indexados: 507
🎉 Merge terminado
✅ Tasks con contexts agregados: 507
❌ Tasks sin match de contexts: 0
📄 Archivo generado: rag_taskAC_with_answerability_and_contexts.jsonl


## **Generación con Modelo GPT-4.1**

In [ ]:
import json
from openai import OpenAI
from google.colab import userdata

# ==========================
#   OBTENER API KEY SECRETA
# ==========================
api_key = userdata.get("OPENAI_API_KEY")
if api_key is None:
    raise ValueError("No se encontró OPENAI_API_KEY en Colab Secrets")

# ==========================
#   CLIENTE OPENAI
# ==========================
client = OpenAI(api_key=api_key)

# ==========================
#   PREPROCESAMIENTO (MT-RAG / TASK C)
# ==========================
def get_current_question(task):
    """
    La pregunta real es el último mensaje del usuario
    """
    msgs = task.get("input", [])
    for m in reversed(msgs):
        if m.get("speaker") == "user" and m.get("text"):
            return m["text"]
    return ""

def format_conversation_history(task):
    """
    Historial completo de la conversación
    """
    msgs = task.get("input", [])
    lines = []
    for m in msgs:
        speaker = (m.get("speaker") or "unknown").upper()
        text = m.get("text") or ""
        lines.append(f"{speaker}: {text}")
    return "\n".join(lines)

def format_contexts(task):
    ctxs = task.get("contexts", [])
    parts = []
    for i, c in enumerate(ctxs, start=1):
        text = c.get("text") or ""
        parts.append(f"[Passage {i}] {text}")
    return "\n\n".join(parts)

# ==========================
#   PROMPT (IDÉNTICO, SOLO SIN VARIABLE)
# ==========================
def generate_response(task):
    current_question = get_current_question(task)
    conversation_history = format_conversation_history(task)
    contexts_text = format_contexts(task)

    prompt = f"""
You are an AI assistant participating in a shared task where the goal is to generate responses that closely match the target answers in the dataset.

Your SINGLE objective:
→ Produce responses that match the *style, content, structure, and level of detail* of the target answers.

Follow these rules STRICTLY:

1. STYLE & TONE
   - Neutral, factual, concise.
   - No conversational fillers like “Yes,” “No,” “Sure,” etc.
   - No opinions, no creativity, no storytelling.

2. LEVEL OF DETAIL
   - Include only the small factual extensions typically seen in the dataset.
   - If the target answers split values (e.g., “six teams from each conference”), include that.
   - If the target clarifies acronyms (e.g., "NFL = National Football League"), include it.
   - Do NOT add details that the dataset style does not include.

3. CONTEXT USE
   - Use contexts only if relevant.
   - Multi-turn questions: use history only to disambiguate, never restate.

4. ANSWERABILITY
   - If marked unanswerable, respond EXACTLY:
     "I'm sorry, but I don't have the answer to your question."

5. STRUCTURE MATCHING
   - Direct declarative sentences.
   - No leading words (“Yes,” “No,” “In conclusion”).
   - Small clarifications permitted only when dataset consistently includes them.

6. NO HALLUCINATIONS
   - Only state facts present in context or trivial domain facts consistent with dataset behavior.

Now generate the response.

Current Question: {current_question}
Conversation History:
{conversation_history}

Available Contexts:
{contexts_text}

Give the final answer in the exact style described above.
"""

    completion = client.chat.completions.create(
        model="gpt-4.1",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.6,
        max_tokens=256
    )

    return completion.choices[0].message.content.strip()

# ==========================
#   LECTURA / ESCRITURA JSONL
# ==========================
input_path = "rag_taskAC_with_answerability_and_contexts.jsonl"
output_path = "responses_TaskC.jsonl"

with open(input_path, "r", encoding="utf-8") as fin, \
     open(output_path, "w", encoding="utf-8") as fout:

    for idx, line in enumerate(fin):
        task = json.loads(line)

        gen = generate_response(task)

        task["predictions"] = [{"text": gen}]
        fout.write(json.dumps(task, ensure_ascii=False) + "\n")

        print(f"=== TASK {task.get('task_id', idx)} ===")
        print(f"PREGUNTA ACTUAL: {get_current_question(task)}")
        print(f"RESPUESTA GENERADA: {gen}")
        print("-" * 80)

print("✅ responses_TaskC.jsonl generado correctamente.")


=== TASK 18ef26058d321c5d96ca3ebf8117789e<::>7 ===
PREGUNTA ACTUAL: I mean current EV's battery does not stand for a used car market...how do you think?
RESPUESTA GENERADA: EV (electric vehicle) batteries are warrantied for 8 years or 100,000 miles by law. This warranty is the minimum, and some batteries have lasted much longer, with examples of over 300,000 miles. As electric cars become more widespread, there may be a second-life market for used batteries in static storage, which could offset the cost of new batteries.
--------------------------------------------------------------------------------
=== TASK b3b321e9ea81d1d90e528f85fff72d63<::>8 ===
PREGUNTA ACTUAL: I heard the toolchain is not available in South America.
RESPUESTA GENERADA: Toolchains are available in Sao Paulo, which is in South America.
--------------------------------------------------------------------------------
=== TASK 72013190593248ab20cca034f21ce38a<::>4 ===
PREGUNTA ACTUAL: Are those the only steps?
RESPUE

In [ ]:
import json

input_path = "IIMAS-RAG_taskC.jsonl"
output_path = "IIMAS-RAG_taskC_format_fixed.jsonl"

with open(input_path, "r", encoding="utf-8") as fin, \
     open(output_path, "w", encoding="utf-8") as fout:

    for line_no, line in enumerate(fin, start=1):
        task = json.loads(line)

        # -------- task_id --------
        if "task_id" in task:
            task["task_id"] = str(task["task_id"])

        # -------- input (historial) --------
        if "input" not in task or not isinstance(task["input"], list):
            task["input"] = []

        # -------- contexts --------
        if "contexts" not in task or not isinstance(task["contexts"], list):
            task["contexts"] = []

        for ctx in task["contexts"]:
            if "document_id" in ctx:
                ctx["document_id"] = str(ctx["document_id"])

            # score es obligatorio para rag_taskc
            if "score" not in ctx or not isinstance(ctx["score"], (int, float)):
                ctx["score"] = 0.0

            if "text" not in ctx:
                ctx["text"] = ""

        # -------- predictions --------
        preds = task.get("predictions")
        if not isinstance(preds, list):
            task["predictions"] = []

        else:
            fixed_preds = []
            for p in preds:
                if isinstance(p, dict) and "text" in p:
                    fixed_preds.append({"text": str(p["text"])})
            task["predictions"] = fixed_preds

        fout.write(json.dumps(task, ensure_ascii=False) + "\n")

print("✅ Archivo normalizado sin regenerar respuestas")
print(f"📄 Archivo listo para checker: {output_path}")

✅ Archivo normalizado sin regenerar respuestas
📄 Archivo listo para checker: IIMAS-RAG_taskC_format_fixed.jsonl


In [ ]:
import json
import os
from collections import Counter

# ==========================
#   CONFIGURACIÓN DIRECTA
# ==========================
INPUT_FILE = "IIMAS-RAG_taskC.jsonl"
PREDICTION_FILE = "IIMAS-RAG_taskC_format_fixed.jsonl"
MODE = "rag_taskc"
MAX_CONTEXTS = 10

# ==========================
#   UTILIDADES
# ==========================
def check_file_size(file_path, max_size_mb=20):
    file_size_bytes = os.path.getsize(file_path)
    file_size_mb = file_size_bytes / (1024 * 1024)
    print(f"File size ({file_path}): {file_size_mb:.2f} MB")
    return file_size_mb <= max_size_mb


def validate_json(line, line_no, errors):
    try:
        return json.loads(line)
    except json.JSONDecodeError as e:
        errors.append(f"[Line {line_no}] Invalid JSON: {e}")
        return None


def validate_taskc(item, line_no, errors):
    required = ["task_id", "Collection", "input", "contexts", "predictions"]

    for field in required:
        if field not in item:
            errors.append(f"[Line {line_no}] Missing required field '{field}'")

    if "task_id" in item and not isinstance(item["task_id"], str):
        errors.append(f"[Line {line_no}] 'task_id' must be a string")

    if "Collection" in item and not isinstance(item["Collection"], str):
        errors.append(f"[Line {line_no}] 'Collection' must be a string")

    if "input" in item and not isinstance(item["input"], list):
        errors.append(f"[Line {line_no}] 'input' must be a list")

    if "contexts" in item and not isinstance(item["contexts"], list):
        errors.append(f"[Line {line_no}] 'contexts' must be a list")

    if "predictions" in item and not isinstance(item["predictions"], list):
        errors.append(f"[Line {line_no}] 'predictions' must be a list")


def validate_contexts(item, line_no, errors):
    contexts = item.get("contexts", [])

    if len(contexts) > MAX_CONTEXTS:
        errors.append(
            f"[Line {line_no}] Too many contexts ({len(contexts)}), max {MAX_CONTEXTS}"
        )

    for i, ctx in enumerate(contexts):
        if not isinstance(ctx, dict):
            errors.append(f"[Line {line_no}] contexts[{i}] must be an object")
            continue

        if "document_id" not in ctx:
            errors.append(f"[Line {line_no}] contexts[{i}] missing 'document_id'")
        elif not isinstance(ctx["document_id"], str):
            errors.append(f"[Line {line_no}] contexts[{i}].document_id must be string")

        if "score" not in ctx:
            errors.append(f"[Line {line_no}] contexts[{i}] missing 'score'")
        elif not isinstance(ctx["score"], (int, float)):
            errors.append(f"[Line {line_no}] contexts[{i}].score must be numeric")

        if "text" not in ctx:
            errors.append(f"[Line {line_no}] contexts[{i}] missing 'text'")
        elif not isinstance(ctx["text"], str):
            errors.append(f"[Line {line_no}] contexts[{i}].text must be string")


def validate_predictions(item, line_no, errors):
    for i, p in enumerate(item.get("predictions", [])):
        if not isinstance(p, dict):
            errors.append(f"[Line {line_no}] predictions[{i}] must be an object")
            continue
        if "text" not in p:
            errors.append(f"[Line {line_no}] predictions[{i}] missing 'text'")
        elif not isinstance(p["text"], str):
            errors.append(f"[Line {line_no}] predictions[{i}].text must be string")


def compare_task_ids(input_file, prediction_file):
    def read_ids(path):
        ids = []
        with open(path, "r", encoding="utf-8") as f:
            for line in f:
                try:
                    obj = json.loads(line)
                    if isinstance(obj.get("task_id"), str):
                        ids.append(obj["task_id"])
                except:
                    pass
        return ids

    input_ids = read_ids(input_file)
    output_ids = read_ids(prediction_file)

    errors = []

    if len(input_ids) != len(output_ids):
        errors.append(
            f"Mismatch in number of instances: input={len(input_ids)}, output={len(output_ids)}"
        )

    missing = set(input_ids) - set(output_ids)
    extra = set(output_ids) - set(input_ids)

    for tid in missing:
        errors.append(f"Missing task_id '{tid}' in prediction file")

    for tid in extra:
        errors.append(f"Extra task_id '{tid}' in prediction file")

    return errors


# ==========================
#   EJECUCIÓN DEL CHECKER
# ==========================
print("🔍 Checking file size...")
check_file_size(PREDICTION_FILE)

errors = []
errors.extend(compare_task_ids(INPUT_FILE, PREDICTION_FILE))

with open(PREDICTION_FILE, "r", encoding="utf-8") as f:
    for line_no, line in enumerate(f, start=1):
        item = validate_json(line, line_no, errors)
        if item is None:
            continue
        validate_taskc(item, line_no, errors)
        validate_contexts(item, line_no, errors)
        validate_predictions(item, line_no, errors)

print("\n--- FORMAT CHECK RESULTS ---")

if not errors:
    print("✅ Format is valid for the eval script.")
else:
    print(f"❌ Found {len(errors)} issue(s):")
    for e in errors:
        print(" -", e)

🔍 Checking file size...
File size (IIMAS-RAG_taskC_format_fixed.jsonl): 3.97 MB

--- FORMAT CHECK RESULTS ---
✅ Format is valid for the eval script.


In [ ]:
import json
import os

# ==========================
#   CONFIGURACIÓN
# ==========================
PREDICTION_FILE = "responses_taskB.json"
MAX_PREDICTIONS = 1   # Task B normalmente espera 1 respuesta por turno

# ==========================
#   UTILIDADES
# ==========================
def check_file_size(file_path, max_size_mb=20):
    file_size_bytes = os.path.getsize(file_path)
    file_size_mb = file_size_bytes / (1024 * 1024)
    print(f"File size ({file_path}): {file_size_mb:.2f} MB")
    return file_size_mb <= max_size_mb


def validate_json(line, line_no, errors):
    try:
        return json.loads(line)
    except json.JSONDecodeError as e:
        errors.append(f"[Line {line_no}] Invalid JSON: {e}")
        return None


def validate_taskb(item, line_no, errors):
    required = ["task_id", "Collection", "input", "predictions"]

    for field in required:
        if field not in item:
            errors.append(f"[Line {line_no}] Missing required field '{field}'")

    if "task_id" in item and not isinstance(item["task_id"], str):
        errors.append(f"[Line {line_no}] 'task_id' must be a string")

    if "Collection" in item and not isinstance(item["Collection"], str):
        errors.append(f"[Line {line_no}] 'Collection' must be a string")

    if "input" in item:
        if not isinstance(item["input"], list):
            errors.append(f"[Line {line_no}] 'input' must be a list")
        else:
            for i, turn in enumerate(item["input"]):
                if not isinstance(turn, dict):
                    errors.append(f"[Line {line_no}] input[{i}] must be an object")
                else:
                    if "speaker" not in turn or "text" not in turn:
                        errors.append(
                            f"[Line {line_no}] input[{i}] must contain 'speaker' and 'text'"
                        )

    if "predictions" in item:
        if not isinstance(item["predictions"], list):
            errors.append(f"[Line {line_no}] 'predictions' must be a list")
        elif len(item["predictions"]) > MAX_PREDICTIONS:
            errors.append(
                f"[Line {line_no}] Too many predictions ({len(item['predictions'])}), max {MAX_PREDICTIONS}"
            )


def validate_predictions(item, line_no, errors):
    for i, p in enumerate(item.get("predictions", [])):
        if not isinstance(p, dict):
            errors.append(f"[Line {line_no}] predictions[{i}] must be an object")
            continue
        if "text" not in p:
            errors.append(f"[Line {line_no}] predictions[{i}] missing 'text'")
        elif not isinstance(p["text"], str):
            errors.append(f"[Line {line_no}] predictions[{i}].text must be string")
        elif not p["text"].strip():
            errors.append(f"[Line {line_no}] predictions[{i}].text is empty")


# ==========================
#   EJECUCIÓN DEL CHECKER
# ==========================
print("🔍 Checking file size...")
check_file_size(PREDICTION_FILE)

errors = []

with open(PREDICTION_FILE, "r", encoding="utf-8") as f:
    for line_no, line in enumerate(f, start=1):
        item = validate_json(line, line_no, errors)
        if item is None:
            continue
        validate_taskb(item, line_no, errors)
        validate_predictions(item, line_no, errors)

print("\n--- TASK B FORMAT CHECK RESULTS ---")

if not errors:
    print("✅ Format is valid for Task B evaluation.")
else:
    print(f"❌ Found {len(errors)} issue(s):")
    for e in errors:
        print(" -", e)

Se truncaron las últimas líneas 5000 del resultado de transmisión.
 - [Line 43858] Invalid JSON: Extra data: line 1 column 23 (char 22)
 - [Line 43859] Invalid JSON: Expecting value: line 1 column 9 (char 8)
 - [Line 43860] Invalid JSON: Expecting value: line 1 column 7 (char 6)
 - [Line 43861] Invalid JSON: Expecting value: line 1 column 5 (char 4)
 - [Line 43862] Invalid JSON: Extra data: line 1 column 17 (char 16)
 - [Line 43863] Invalid JSON: Extra data: line 1 column 18 (char 17)
 - [Line 43864] Invalid JSON: Expecting property name enclosed in double quotes: line 2 column 1 (char 8)
 - [Line 43865] Invalid JSON: Extra data: line 1 column 15 (char 14)
 - [Line 43866] Invalid JSON: Expecting value: line 1 column 7 (char 6)
 - [Line 43867] Invalid JSON: Expecting value: line 1 column 5 (char 4)
 - [Line 43868] Invalid JSON: Expecting value: line 1 column 3 (char 2)
 - [Line 43869] Invalid JSON: Expecting property name enclosed in double quotes: line 2 column 1 (char 4)
 - [Line 4387

In [ ]:
# ==========================
# CONFIGURACIÓN (EDITA SOLO AQUÍ)
# ==========================
INPUT_FILE = "reference_taskB.jsonl"        # archivo input oficial
PREDICTION_FILE = "responses_taskB_fixed.jsonl"  # tus respuestas
MODE = "generation_taskb"  # <-- IMPORTANTE para Task B

# ==========================
# CARGAR FORMAT CHECKER OFICIAL
# ==========================
import json
import os
from collections import Counter

# Copia literal del checker (sin argparse)
MAX_CONTEXTS = 10
EMPTY_CONTEXT_LINES = []

def check_file_size(file_path, max_size_mb=20):
    file_size_bytes = os.path.getsize(file_path)
    file_size_mb = file_size_bytes / (1024 * 1024)
    print(f"File size: {file_size_mb:.2f} MB")
    if file_size_mb > max_size_mb:
        print(f"❌ File exceeds {max_size_mb} MB limit.")
        return False
    print("✅ File size is within the limit.")
    return True


def validate_json(line, line_no, errors):
    try:
        return json.loads(line)
    except json.JSONDecodeError as e:
        errors.append(f"[Line {line_no}] Invalid JSON: {e}")
        return None


def validate_required_fields_generation(item, line_no, errors):
    required = ["task_id", "input", "contexts", "predictions"]

    for field in required:
        if field not in item:
            errors.append(f"[Line {line_no}] Missing required field '{field}'")

    if "task_id" in item and not isinstance(item["task_id"], str):
        errors.append(f"[Line {line_no}] 'task_id' must be a string")

    if "input" in item and not isinstance(item["input"], list):
        errors.append(f"[Line {line_no}] 'input' must be a list")

    if "contexts" in item and not isinstance(item["contexts"], list):
        errors.append(f"[Line {line_no}] 'contexts' must be a list")

    if "predictions" in item and not isinstance(item["predictions"], list):
        errors.append(f"[Line {line_no}] 'predictions' must be a list")


def validate_contexts(item, line_no, errors):
    contexts = item.get("contexts", [])

    if isinstance(contexts, list) and len(contexts) == 0:
        EMPTY_CONTEXT_LINES.append(line_no)

    if len(contexts) > MAX_CONTEXTS:
        errors.append(
            f"[Line {line_no}] Too many contexts ({len(contexts)}), max {MAX_CONTEXTS}"
        )

    for i, ctx in enumerate(contexts):
        if not isinstance(ctx, dict):
            errors.append(f"[Line {line_no}] contexts[{i}] must be an object")
            continue

        if "document_id" not in ctx:
            errors.append(f"[Line {line_no}] contexts[{i}] missing 'document_id'")
        elif not isinstance(ctx["document_id"], str):
            errors.append(f"[Line {line_no}] contexts[{i}].document_id must be string")

        if "text" not in ctx:
            errors.append(f"[Line {line_no}] contexts[{i}] missing 'text'")
        elif not isinstance(ctx["text"], str):
            errors.append(f"[Line {line_no}] contexts[{i}].text must be string")


def validate_predictions(item, line_no, errors):
    for i, p in enumerate(item.get("predictions", [])):
        if not isinstance(p, dict):
            errors.append(f"[Line {line_no}] predictions[{i}] must be an object")
            continue
        if "text" not in p:
            errors.append(f"[Line {line_no}] predictions[{i}] missing 'text'")
        elif not isinstance(p["text"], str):
            errors.append(f"[Line {line_no}] predictions[{i}].text must be string")


def compare_task_ids(input_file, prediction_file):
    def read_ids(path):
        ids = []
        with open(path, "r", encoding="utf-8") as f:
            for line in f:
                try:
                    obj = json.loads(line)
                    if isinstance(obj.get("task_id"), str):
                        ids.append(obj["task_id"])
                except:
                    pass
        return ids

    input_ids = read_ids(input_file)
    output_ids = read_ids(prediction_file)

    errors = []

    if len(input_ids) != len(output_ids):
        errors.append(
            f"Mismatch in number of instances: input={len(input_ids)}, output={len(output_ids)}"
        )

    missing = set(input_ids) - set(output_ids)
    extra = set(output_ids) - set(input_ids)

    for tid in missing:
        errors.append(f"Missing task_id '{tid}' in prediction file")

    for tid in extra:
        errors.append(f"Extra task_id '{tid}' in prediction file")

    return errors


# ==========================
# EJECUCIÓN DEL CHECKER
# ==========================
print("🔍 Checking file size...")
check_file_size(PREDICTION_FILE)

errors = []
errors.extend(compare_task_ids(INPUT_FILE, PREDICTION_FILE))

with open(PREDICTION_FILE, "r", encoding="utf-8") as f:
    for line_no, line in enumerate(f, start=1):
        item = validate_json(line, line_no, errors)
        if item is None:
            continue
        validate_required_fields_generation(item, line_no, errors)
        validate_contexts(item, line_no, errors)
        validate_predictions(item, line_no, errors)

print("\n--- FORMAT CHECK RESULTS (Task B) ---")

if EMPTY_CONTEXT_LINES:
    print(f"⚠️ Warning: empty contexts on {len(EMPTY_CONTEXT_LINES)} lines")

if not errors:
    print("✅ Format is valid for the official eval script.")
else:
    print(f"❌ Found {len(errors)} issue(s):")
    for e in errors:
        print(" -", e)


🔍 Checking file size...
File size: 3.50 MB
✅ File size is within the limit.

--- FORMAT CHECK RESULTS (Task B) ---
⚠️ Warning: empty contexts on 130 lines
❌ Found 7 issue(s):
 - [Line 456] Too many contexts (12), max 10
 - [Line 458] Too many contexts (12), max 10
 - [Line 467] Too many contexts (23), max 10
 - [Line 482] Too many contexts (28), max 10
 - [Line 488] Too many contexts (35), max 10
 - [Line 489] Too many contexts (29), max 10
 - [Line 492] Too many contexts (38), max 10


In [ ]:
import json

input_path = "responses_taskB.json"
output_path = "responses_taskB_fixed.jsonl"

with open(input_path, "r", encoding="utf-8") as f:
    data = json.load(f)   # <- aquí está la clave

print(f"Objetos leídos: {len(data)}")

with open(output_path, "w", encoding="utf-8") as f:
    for obj in data:
        f.write(json.dumps(obj, ensure_ascii=False) + "\n")

print("✅ Archivo JSONL generado:", output_path)


Objetos leídos: 507
✅ Archivo JSONL generado: responses_taskB_fixed.jsonl


In [ ]:
!python format_checker.py \
  --input_file reference_taskB.jsonl \
  --prediction_file responses_taskB_fixed.jsonl \
  --mode generation_taskb



File size: 3.50 MB
File size is within the limit.

--- Format Check Results ---
Found 1 warning(s):
 - 'contexts' is empty on 130 line(s): [3, 19, 28, 34, 48, 53, 55, 58, 60, 61, 65, 76, 83, 90, 94, 97, 98, 102, 105, 112, 114, 117, 118, 123, 127, 131, 136, 140, 141, 142, 146, 147, 148, 151, 152, 154, 161, 166, 171, 173, 177, 180, 182, 187, 198, 207, 209, 210, 211, 212, 219, 221, 224, 229, 237, 241, 244, 245, 246, 260, 263, 266, 269, 277, 278, 285, 292, 306, 315, 322, 324, 326, 330, 337, 338, 342, 350, 354, 355, 356, 360, 361, 375, 379, 380, 381, 385, 389, 393, 398, 403, 404, 409, 412, 419, 420, 422, 469, 470, 471, 472, 473, 474, 475, 476, 477, 478, 479, 480, 481, 483, 484, 485, 486, 487, 491, 493, 494, 495, 496, 497, 498, 499, 500, 501, 502, 503, 504, 505, 506]
Found 1187 issue(s):
 - [Line 1] contexts[0] missing 'score'
 - [Line 1] contexts[1] missing 'score'
 - [Line 1] contexts[2] missing 'score'
 - [Line 1] contexts[3] missing 'score'
 - [Line 2] contexts[0] missing 'score'
 - [Lin